# BCC Fe 3×3×3: stochastic noncollinear QE inputs at 4000 K

This notebook builds a **54-atom BCC Fe** noncollinear magnetic-SQS calculation using:

- the TDEP force constants fitted at `a = 2.55 Å`, `4000 K`;
- a classical stochastic displacement generated by TDEP `canonical_configuration`;
- a zero-net noncollinear magnetic SQS with 54 unique QE species;
- Maxwell–Boltzmann velocities in QE Rydberg atomic units, with COM drift removed and exact 4000 K rescaling;
- the SCF settings supplied for unconstrained noncollinear preconvergence.

It writes a stage-1 SCF input and a stage-2 MD input. The TDEP real-space cutoff (`4.4167 Å`) is larger than half the `3×3×3` box length (`3.825 Å`), so this smaller-cell stochastic configuration is useful as a requested starting configuration but should be checked for finite-size effects before quantitative ensemble analysis.

In [1]:
from pathlib import Path
import json
import re
import subprocess
import sys

import numpy as np

WORKSPACE = Path.cwd().resolve()
if not (WORKSPACE / 'IronCoreMD').exists():
    WORKSPACE = Path.cwd().resolve().parents[1]

CODES = WORKSPACE / 'IronCoreMD' / 'codes'
if str(CODES) not in sys.path:
    sys.path.insert(0, str(CODES))

from prepare_latest_bcc_qe_input import (
    MAGNETIC_MODE_NONCOLLINEAR_RANDOM,
    SPIN_MODE_MAGNETIC_SQS,
    magnetic_neighbor_shells,
    magnetic_shell_correlations,
    prepare_bcc_qe_input,
)
from generate_qe_maxwell_velocities import (
    parse_atomic_velocities,
    read_qe_species_sequence,
    temperature_from_velocities_au,
)

TDEP_FOLDER = WORKSPACE / 'dataset' / 'bcc' / 'non-mag' / 'tdep_2.55_4000K'
OUTPUT_DIR = WORKSPACE / 'prepared_qe_inputs' / 'noncollinear_3x3x3_4000K'
STOCHASTIC_DIR = OUTPUT_DIR / 'tdep_stochastic'
STOCHASTIC_QE = STOCHASTIC_DIR / 'qe_conf0001'
INTERMEDIATE_NPZ = OUTPUT_DIR / 'bcc_3x3x3_stochastic_4000K.npz'
STAGE1_SCF = OUTPUT_DIR / 'Fe_bcc_3x3x3_noncollinear_SQS_stage1_unconstrained_scf.in'
STAGE2_MD = OUTPUT_DIR / 'Fe_bcc_3x3x3_noncollinear_SQS_stage2_md_4000K_with_velocities.in'
SUMMARY_JSON = OUTPUT_DIR / 'generation_summary.json'

TEMPERATURE_K = 4000.0
LATTICE_PARAMETER_ANG = 2.55
SUPERCELL = (3, 3, 3)
SPIN_SEED = 20260718
VELOCITY_SEED = 400054
STARTING_MAGNITUDE = 0.15
MINIMUM_DISTANCE_RATIO = 0.75
N_STOCHASTIC_CANDIDATES = 20
REGENERATE_STOCHASTIC = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)

Output: /Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K


In [2]:
if REGENERATE_STOCHASTIC or not STOCHASTIC_QE.exists():
    command = [
        sys.executable,
        str(CODES / 'generate_stochastic_bcc_configs.py'),
        str(TDEP_FOLDER),
        '--temperature', str(TEMPERATURE_K),
        '--nconf', str(N_STOCHASTIC_CANDIDATES),
        '--format', 'qe',
        '--supercell', *(str(value) for value in SUPERCELL),
        '--output-dir', str(STOCHASTIC_DIR),
    ]
    subprocess.run(command, cwd=WORKSPACE, check=True)

candidate_files = sorted(STOCHASTIC_DIR.glob('qe_conf*'))
assert len(candidate_files) == N_STOCHASTIC_CANDIDATES
print(f'Generated {len(candidate_files)} stochastic candidates')

 ... reading infiles
 ... remapped fc
   no.   +/-      ek(K)          ep(K)          <ek/ep>         T(K)          <T>(K)      <msd>(A)
    1     +     4076.61532     4296.38692        0.94885     4186.50112     4186.50112       0.33862953
    2     +     4523.00009     2992.10659        1.17989     3757.55334     3972.02723       0.32414368
    3     +     3263.59034     3412.64930        1.10859     3338.11982     3760.72476       0.32226220
    4     +     4172.62739     4213.05682        1.07521     4192.84211     3868.75410       0.33173663
    5     +     4203.74746     3482.93723        1.10015     3843.34234     3863.67175       0.32318112
    6     +     4272.77376     3462.10233        1.12137     3867.43804     3864.29946       0.32367043
    7     +     3939.46193     3846.94041        1.10681     3893.20117     3868.42828       0.32935249
    8     +     3579.96204     3429.19160        1.09941     3504.57682     3822.94685       0.32761429
    9     +     3093.79436     

In [3]:
BOHR_TO_ANG = 0.529177210903

def parse_tdep_qe_configuration(path: Path):
    lines = path.read_text().splitlines()
    natoms = int(next(line.split('=')[1] for line in lines if 'nat' in line and '=' in line))
    cell_start = next(i for i, line in enumerate(lines) if line.strip().upper().startswith('CELL_PARAMETERS'))
    position_start = next(i for i, line in enumerate(lines) if line.strip().upper().startswith('ATOMIC_POSITIONS'))
    cell_bohr = np.asarray([[float(value) for value in lines[cell_start + j].split()[:3]] for j in range(1, 4)])
    positions_bohr = np.asarray([[float(value) for value in lines[position_start + j].split()[1:4]] for j in range(1, natoms + 1)])
    cell_ang = cell_bohr * BOHR_TO_ANG
    positions_frac = (positions_bohr @ np.linalg.inv(cell_bohr)) % 1.0
    return cell_ang, positions_frac

def read_direct_poscar(path: Path):
    lines = path.read_text().splitlines()
    scale = float(lines[1].split()[0])
    cell = scale * np.asarray([[float(value) for value in lines[index].split()[:3]] for index in range(2, 5)])
    natoms = int(lines[6].split()[0])
    positions = np.asarray([[float(value) for value in lines[8 + index].split()[:3]] for index in range(natoms)])
    return cell, positions

ideal_cell_ang, ideal_frac = read_direct_poscar(STOCHASTIC_DIR / 'infile.ssposcar')

def minimum_pair_distance(positions_frac: np.ndarray, cell_ang: np.ndarray) -> float:
    distances = []
    for first in range(len(positions_frac)):
        pair_delta = positions_frac[first + 1:] - positions_frac[first]
        pair_delta -= np.rint(pair_delta)
        distances.extend(np.linalg.norm(pair_delta @ cell_ang, axis=1))
    return float(np.min(distances))

ideal_nearest_neighbor_ang = float(np.sqrt(3.0) * LATTICE_PARAMETER_ANG / 2.0)
candidate_records = []
for candidate_path in candidate_files:
    candidate_cell, candidate_frac = parse_tdep_qe_configuration(candidate_path)
    candidate_minimum = minimum_pair_distance(candidate_frac, candidate_cell)
    candidate_records.append((candidate_minimum / ideal_nearest_neighbor_ang, candidate_path, candidate_cell, candidate_frac))
minimum_distance_ratio, STOCHASTIC_QE, cell_ang, displaced_frac = max(candidate_records, key=lambda record: record[0])
minimum_pair_distance_ang = minimum_distance_ratio * ideal_nearest_neighbor_ang
assert minimum_distance_ratio >= MINIMUM_DISTANCE_RATIO
print(f'Selected {STOCHASTIC_QE.name} from {len(candidate_records)} candidates')
natoms = len(displaced_frac)
assert natoms == 54
assert np.allclose(cell_ang, ideal_cell_ang, atol=1.0e-8)

np.savez_compressed(
    INTERMEDIATE_NPZ,
    input_cell_parameters=cell_ang.astype(np.float32),
    input_cell_unit=np.asarray('angstrom', dtype='U16'),
    symbols=np.full(natoms, 'Fe', dtype='U2'),
    species=np.full(natoms, 'Fe', dtype='U2'),
    positions=displaced_frac[None, :, :].astype(np.float32),
    positions_unit=np.asarray(['crystal'], dtype='U16'),
    forces_ry_au=np.zeros((1, natoms, 3), dtype=np.float32),
    initial_positions_alat=ideal_frac.astype(np.float32),
    initial_cell_alat=np.eye(3, dtype=np.float32),
    cell_parameters=cell_ang[None, :, :].astype(np.float32),
    cell_parameters_unit=np.asarray(['angstrom'], dtype='U16'),
    iteration=np.asarray([0], dtype=np.int32),
    time_ps=np.asarray([0.0], dtype=np.float32),
    temperature_K=np.asarray([TEMPERATURE_K], dtype=np.float32),
    pressure_GPa=np.asarray([np.nan], dtype=np.float32),
)

delta_frac = displaced_frac - ideal_frac
delta_frac -= np.rint(delta_frac)
displacements_ang = delta_frac @ cell_ang
rms_displacement_ang = float(np.sqrt(np.mean(np.sum(displacements_ang**2, axis=1))))
print(f'Atoms: {natoms}')
print(f'Cell lengths: {np.linalg.norm(cell_ang, axis=1)} Å')
print(f'RMS stochastic displacement: {rms_displacement_ang:.6f} Å')
print(f'Minimum pair distance: {minimum_pair_distance_ang:.6f} Å ({minimum_distance_ratio:.4f} × ideal NN)')
print('Intermediate NPZ:', INTERMEDIATE_NPZ)

Selected qe_conf0013 from 20 candidates
Atoms: 54
Cell lengths: [7.64999986 7.64999986 7.64999986] Å
RMS stochastic displacement: 0.352039 Å
Minimum pair distance: 1.811829 Å (0.8204 × ideal NN)
Intermediate NPZ: /Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/bcc_3x3x3_stochastic_4000K.npz


In [4]:
result = prepare_bcc_qe_input(
    dataset_dir=INTERMEDIATE_NPZ.parent,
    npz_path=INTERMEDIATE_NPZ,
    output_dir=OUTPUT_DIR,
    frame_index=0,
    temperature_k=TEMPERATURE_K,
    velocity_seed=VELOCITY_SEED,
    spin_seed=SPIN_SEED,
    magnetic_mode=MAGNETIC_MODE_NONCOLLINEAR_RANDOM,
    spin_mode=SPIN_MODE_MAGNETIC_SQS,
    m_abs=STARTING_MAGNITUDE,
    pseudo_dir='./pseudo',
    pseudo_file='Fe.pbe-spn-kjpaw_psl.1.0.0.UPF',
    dt_au=20.670,
    nstep=400,
    ecutwfc=100.0,
    ecutrho=496.0,
    degauss=0.02,
    k_grid=(2, 2, 2, 0, 0, 0),
    constrained_magnetization=False,
    mixing_beta=0.01,
    mixing_mode='local-TF',
    mixing_ndim=12,
    diagonalization='cg',
    remove_com_drift=True,
    rescale_exact=True,
    nosym=True,
)
result.as_dict()

Magnetic SQS shell correlations: shell1=-0.00016404 shell2=+0.00028715 shell3=+0.00009372 shell4=+0.00038181 shell5=+0.00278488
Magnetic SQS objective: 1.603891959923e-06
Number of atoms: 54
Number of species: 54
First 10 labels: ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10']
Last 10 labels: ['A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54']
Position and velocity labels match exactly: True


{'npz_path': '/Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/bcc_3x3x3_stochastic_4000K.npz',
 'output_dir': '/Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K',
 'frame_index': 0,
 'natoms': 54,
 'structure_tag': 'bcc_3x3x3',
 'target_temperature_k': 4000.0,
 'measured_velocity_temperature_k': 3999.999999999999,
 'qe_input_base': '/Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/Fe_bcc_3x3x3_latest_noncollinear_paramagnetic_4000K_base.in',
 'qe_input_final': '/Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/Fe_bcc_3x3x3_latest_noncollinear_paramagnetic_4000K_with_velocities.in',
 'velocity_block': '/Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/Fe_bcc_3x3x3_latest_noncollinear_paramagnetic_atomic_velocities_4000K.txt',
 'spin_vectors': '/Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/Fe_bcc_3x3x3_latest_noncollinear_paramagnetic_spin

In [5]:
def remove_namelist(text: str, name: str) -> str:
    return re.sub(rf'(?ims)^\s*&{name}\b.*?^\s*/\s*\n?', '', text)

def make_stage1_scf(md_base_text: str) -> str:
    text = re.sub(r"(?mi)^(\s*)calculation\s*=.*$", r"\1calculation='scf'", md_base_text)
    text = re.sub(r"(?mi)^(\s*)outdir\s*=.*$", r"\1outdir='./scf_preconverge'", text)
    text = re.sub(r'(?mi)^\s*(?:dt|nstep)\s*=.*\n?', '', text)
    return remove_namelist(text, 'ions')

def make_stage2_md(md_text: str) -> str:
    text = re.sub(r"(?mi)^(\s*)outdir\s*=.*$", r"\1outdir='./scf_preconverge'", md_text)
    marker = ' &electrons\n'
    replacement = marker + "    startingpot='file'\n    startingwfc='file'\n"
    if "startingpot='file'" not in text:
        text = text.replace(marker, replacement, 1)
    return text

stage1_text = make_stage1_scf(Path(result.qe_input_base).read_text())
stage2_text = make_stage2_md(Path(result.qe_input_final).read_text())
STAGE1_SCF.write_text(stage1_text)
STAGE2_MD.write_text(stage2_text)
print('Stage 1:', STAGE1_SCF)
print('Stage 2:', STAGE2_MD)

Stage 1: /Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/Fe_bcc_3x3x3_noncollinear_SQS_stage1_unconstrained_scf.in
Stage 2: /Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/Fe_bcc_3x3x3_noncollinear_SQS_stage2_md_4000K_with_velocities.in


In [6]:
def card_labels(path: Path, card: str, nfields: int):
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip().upper().startswith(card))
    labels = []
    for line in lines[start + 1:]:
        fields = line.split()
        if len(fields) < nfields:
            if labels:
                break
            continue
        labels.append(fields[0])
    return labels

species_labels = card_labels(STAGE2_MD, 'ATOMIC_SPECIES', 3)
position_labels = card_labels(STAGE2_MD, 'ATOMIC_POSITIONS', 4)
velocity_labels = card_labels(STAGE2_MD, 'ATOMIC_VELOCITIES', 4)
parsed_velocity_labels, velocities_au = parse_atomic_velocities(STAGE2_MD.read_text().splitlines())
mass_labels, masses_amu = read_qe_species_sequence(STAGE2_MD)
measured_temperature = temperature_from_velocities_au(velocities_au, masses_amu, remove_com=True)

spin_vectors = np.loadtxt(result.spin_vectors, comments='#')[:, -3:]
spin_directions = spin_vectors / np.linalg.norm(spin_vectors, axis=1)[:, None]
shells = magnetic_neighbor_shells(ideal_frac, cell_ang)
shell_correlations = magnetic_shell_correlations(spin_directions, shells)
net_spin = np.sum(spin_vectors, axis=0)

assert len(species_labels) == len(position_labels) == len(velocity_labels) == 54
assert species_labels == position_labels == velocity_labels == parsed_velocity_labels == mass_labels
assert abs(measured_temperature - TEMPERATURE_K) < 1.0e-8
assert np.linalg.norm(net_spin) < 1.0e-10
assert "calculation='scf'" in STAGE1_SCF.read_text()
assert '&ions' not in STAGE1_SCF.read_text().lower()
assert 'ATOMIC_VELOCITIES' not in STAGE1_SCF.read_text()
assert "startingpot='file'" in STAGE2_MD.read_text()
assert "startingwfc='file'" in STAGE2_MD.read_text()

summary = {
    'source_tdep_folder': str(TDEP_FOLDER),
    'lattice_parameter_ang': LATTICE_PARAMETER_ANG,
    'temperature_K': TEMPERATURE_K,
    'supercell': list(SUPERCELL),
    'natoms': natoms,
    'ntyp': natoms,
    'rms_stochastic_displacement_ang': rms_displacement_ang,
    'stochastic_candidates_generated': N_STOCHASTIC_CANDIDATES,
    'selected_stochastic_configuration': STOCHASTIC_QE.name,
    'minimum_pair_distance_ang': minimum_pair_distance_ang,
    'minimum_distance_ratio': minimum_distance_ratio,
    'velocity_seed': VELOCITY_SEED,
    'measured_velocity_temperature_K': measured_temperature,
    'spin_seed': SPIN_SEED,
    'starting_magnetization': STARTING_MAGNITUDE,
    'net_starting_spin': [float(value) for value in net_spin],
    'magnetic_sqs_shell_correlations': [float(value) for value in shell_correlations],
    'stage1_scf': str(STAGE1_SCF),
    'stage2_md': str(STAGE2_MD),
    'finite_size_warning': 'TDEP cutoff 4.4167 Å exceeds half the 3x3x3 box length 3.825 Å.',
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2) + '\n')
summary

{'source_tdep_folder': '/Users/dajuarez4/Documents/Fe/dataset/bcc/non-mag/tdep_2.55_4000K',
 'lattice_parameter_ang': 2.55,
 'temperature_K': 4000.0,
 'supercell': [3, 3, 3],
 'natoms': 54,
 'ntyp': 54,
 'rms_stochastic_displacement_ang': 0.35203941731533905,
 'stochastic_candidates_generated': 20,
 'selected_stochastic_configuration': 'qe_conf0013',
 'minimum_pair_distance_ang': 1.8118285912141003,
 'minimum_distance_ratio': 0.8204389998924875,
 'velocity_seed': 400054,
 'measured_velocity_temperature_K': 3999.999999999999,
 'spin_seed': 20260718,
 'starting_magnetization': 0.15,
 'net_starting_spin': [-5.551115123125783e-17,
  -1.0408340855860843e-17,
  1.3877787807814457e-16],
 'magnetic_sqs_shell_correlations': [-0.00016403856975207072,
  0.000287145001920301,
  9.371532895395294e-05,
  0.0003818096276495295,
  0.0027848766136803946],
 'stage1_scf': '/Users/dajuarez4/Documents/Fe/prepared_qe_inputs/noncollinear_3x3x3_4000K/Fe_bcc_3x3x3_noncollinear_SQS_stage1_unconstrained_scf.in',

## Run order

1. Run `Fe_bcc_3x3x3_noncollinear_SQS_stage1_unconstrained_scf.in` to preconverge the electronic state.
2. Keep the generated `scf_preconverge/` directory unchanged.
3. Run `Fe_bcc_3x3x3_noncollinear_SQS_stage2_md_4000K_with_velocities.in` from the same working directory. It reads `startingpot='file'` and `startingwfc='file'` from the stage-1 prefix.

The 54 unique species labels are intentional: Quantum ESPRESSO assigns `starting_magnetization`, `angle1`, and `angle2` by species, so one species per atom is required for atom-resolved spin directions.